# Meeting Protocol — Prompt Testing

1. Check what production generated (prompt logs)
2. Load the same transcript locally
3. Run the prompt → get the protocol
4. Edit a prompt block, run again → compare

In [1]:
import sys, json, time, copy, requests
from pathlib import Path
from datetime import datetime

sys.path.insert(0, "..")
from utils.api_client import call_model
from utils.prompt_loader import load_prompt

PROMPTS = load_prompt("../prompts/workflow/prompt_builder.yaml")
print("Prompt blocks loaded:")
for key in PROMPTS:
    print(f"  {key} ({len(PROMPTS[key])} chars)")

Prompt blocks loaded:
  specificity_rules_de (1719 chars)
  specificity_rules_en (1482 chars)
  client_meeting_rules_de (532 chars)
  client_meeting_rules_en (479 chars)
  base_protocol_system_de (302 chars)
  base_protocol_system_en (340 chars)
  structure_analysis_de (910 chars)
  structure_analysis_en (819 chars)
  extraction_from_outline_de (859 chars)
  extraction_from_outline_en (872 chars)
  json_formatting_open_source_de (247 chars)
  json_formatting_open_source_en (180 chars)
  speaker_inference_system (1646 chars)


## 0. Production Logs

See what prompt was sent and what protocol production generated.
Use this to compare against your modified prompts.

In [ ]:
# WORKFLOW_URL = "https://workflow-service.mangowater-31feba87.swedencentral.azurecontainerapps.io"

# response = requests.get(f"{WORKFLOW_URL}/debug/prompt-logs", params={"limit": 50}, timeout=10)

# if response.status_code != 200:
#     print(f"Error {response.status_code} — skip to Section 1")
# else:
#     logs = response.json()
#     if not logs:
#         print("No logs yet. Run a meeting first.")
#     else:
#         print(f"Found {len(logs)} entries\n")
#         for i, entry in enumerate(logs):
#             print(f"[{i}] {entry['pass']}  |  {entry['model']}  |  {entry['prompt_length']} chars  |  {entry['timestamp']}")
#             print(f"     {entry['prompt_preview'][:1200]}...")
#             print()

Error 401 — skip to Section 1


In [ ]:
# # Pick a log entry to inspect
# LOG_INDEX = 0  # <-- change this to pick a different entry

# entry = logs[LOG_INDEX]
# print(f"Pass: {entry['pass']}  |  Model: {entry['model']}  |  {entry['timestamp']}")
# print(f"Prompt: {entry['prompt_length']} chars  |  Response: {entry['response_length']} chars")
# if entry.get("usage"):
#     print(f"Usage: {entry['usage']}")

# # Extract speaker names from the prompt
# prompt_text = entry["prompt_preview"]
# prod_speakers = ""
# for line in prompt_text.split("\n"):
#     if line.strip().startswith("Use ONLY these names:"):
#         prod_speakers = line.split(":", 1)[1].strip()
#         break

# print(f"Speakers: {prod_speakers}")
# print(f"\n=== PRODUCTION PROMPT (system) ===")
# print(prompt_text[:2000])
# print(f"\n=== PRODUCTION OUTPUT (protocol) ===")
# print(entry["response_preview"])

## 1. Load Transcript

Save the transcript from the platform as a `.txt` file in `test_cases/`.
Use the same transcript that production used, so you can compare outputs.

In [4]:
# --- CONFIGURE THESE ---
TRANSCRIPT_FILE = "../test_cases/meeting_08.txt"
LANG = "en"              # "en" or "de"
IS_CLIENT_MEETING = False
MODEL = "gpt-4o"         # "gpt-4o" or "mistral-25b"
""
# Auto-fill speakers from production log if available
try:
    SPEAKER_NAMES = prod_speakers if prod_speakers else "Unknown"
except NameError:
    SPEAKER_NAMES = "Unknown"  # <-- set manually if you skipped Section 0

transcript = Path(TRANSCRIPT_FILE).read_text(encoding="utf-8")
print(f"{TRANSCRIPT_FILE}  |  {len(transcript)} chars  |  {LANG}  |  {MODEL}")
print(f"Speakers: {SPEAKER_NAMES}")
print("---")
print(transcript[:500] + "..." if len(transcript) > 500 else transcript)

../test_cases/meeting_08.txt  |  42835 chars  |  en  |  gpt-4o
Speakers: Unknown
---
0:00
Hallo und herzlich willkommen zu unserem Podcast. Heute sprechen wir über ein Thema, dass viele Menschen begeistert reisen und das Entdecken neuer Orte. Es gibt so viele Gründe, warum wir reisen,
0:10
10 seconds
oder was denkst du, Anna? Ja, genau.
0:14
14 seconds
Reisen ist eine der besten Möglichkeiten, dem Alltag zu entfliehen.
0:18
18 seconds
Aber nicht nur das. Es gibt so viele Dinge, die man lernen kann, wenn man in andere Länder reist. Man lernt neue Kulturen kennen, probiert neue
0:...


## 2. Run the Prompt (Baseline)

Assembles the system prompt from blocks (same as production) and runs it.

In [5]:
def build_system_prompt(prompts, lang, is_client=False):
    """Assemble the system prompt from blocks, same as production."""
    parts = [
        prompts[f"base_protocol_system_{lang}"],
        prompts[f"extraction_from_outline_{lang}"],
        prompts[f"specificity_rules_{lang}"],
    ]
    if is_client:
        parts.append(prompts[f"client_meeting_rules_{lang}"])
    parts.append(prompts[f"json_formatting_open_source_{lang}"])
    return "\n\n".join(parts)


system_prompt = build_system_prompt(PROMPTS, LANG, IS_CLIENT_MEETING)
user_prompt = f"Confirmed speaker names: {SPEAKER_NAMES}\n\nTRANSCRIPT:\n{transcript}"

print(f"System prompt: {len(system_prompt)} chars")
print(f"User prompt: {len(user_prompt)} chars")
print(f"\n=== SYSTEM PROMPT ===")
print(system_prompt)

System prompt: 2880 chars
User prompt: 42881 chars

=== SYSTEM PROMPT ===
You are an expert meeting protocol writer.

CRITICAL INSTRUCTIONS:
1. Respond in the SAME LANGUAGE as the transcript.
2. SPECIFICITY: Use concrete names, numbers, dates, and quotes from the transcript. Avoid generic descriptions.
3. Use the confirmed speaker names below, NOT the speaker IDs from the transcript.
4. Output valid JSON only.


You are an expert meeting protocol writer. You receive two inputs:
1. A structural outline of the meeting (from a prior analysis)
2. The original transcript as reference

Your task: Produce the final meeting protocol in the specified JSON format.

CRITICAL INSTRUCTIONS:
1. LANGUAGE: Respond in the same language as the transcript.
2. The outline is your primary source for structure and topics. The transcript is your source for exact quotes, names, and details.
3. SPECIFICITY: Concrete names, numbers, system names, dates. No generic descriptions.
4. COMPLETENESS: Every topic from

In [6]:
print(f"Running {MODEL}...")
start = time.time()
baseline_output = call_model(system_prompt, user_prompt, model=MODEL, max_tokens=8000)
baseline_duration = round(time.time() - start, 2)
print(f"Done in {baseline_duration}s\n")

try:
    clean = baseline_output.strip()
    if clean.startswith("```"):
        clean = clean.split("\n", 1)[1].rsplit("```", 1)[0]
    baseline_protocol = json.loads(clean)
    print(json.dumps(baseline_protocol, indent=2, ensure_ascii=False))
except json.JSONDecodeError as e:
    print(f"JSON parse error: {e}")
    print(baseline_output)
    baseline_protocol = None

Running gpt-4o...
Done in 11.63s

{
  "meeting_protocol": {
    "title": "Podcast über Reisen und nachhaltiges Reisen",
    "date": "TBD",
    "participants": [
      "Anna",
      "Markus"
    ],
    "summary": "In diesem Podcast diskutierten Anna und Markus über die Faszination des Reisens, die Vorteile, die es mit sich bringt, und wie man sich auf Reisen vorbereitet. Sie sprachen über die kulturellen Erfahrungen, die man beim Reisen sammeln kann, und die Bedeutung von nachhaltigem Reisen. Anna und Markus teilten persönliche Erlebnisse und gaben Tipps, wie man umweltbewusst reisen und nachhaltige Souvenirs finden kann.",
    "action_items": [
      {
        "description": "Anna plant, Japan zu besuchen, um die Kultur und den Mount Fuji zu erleben.",
        "due_date": "TBD"
      },
      {
        "description": "Markus möchte Südamerika, insbesondere Peru und Machu Picchu, besuchen.",
        "due_date": "TBD"
      },
      {
        "description": "Anna und Markus empfehlen, Re

## 3. Edit a Prompt Block and Re-run

1. Use the cell below to see the current content of any block
2. Copy it, modify it, and run again
3. Compare the outputs in Section 4

In [7]:
# See the current content of a block
BLOCK = "specificity_rules_de"  # <-- change to see other blocks
# Options: base_protocol_system_de, extraction_from_outline_de, specificity_rules_de,
#          client_meeting_rules_de, json_formatting_open_source_de (same for _en)

print(f"=== {BLOCK} ===")
print(PROMPTS[BLOCK])

=== specificity_rules_de ===
## Qualitätsregeln – UNBEDINGT EINHALTEN
- VOLLSTÄNDIGKEIT: Gehe das GESAMTE Transkript Abschnitt für Abschnitt durch. Überspringe KEINE Themen.
- MINDESTANZAHL: Extrahiere mindestens 8 Aktionspunkte und mindestens 5 Entscheidungen aus einem Meeting über 30 Minuten. Bei einem 90-Minuten-Meeting erwarte ich 15+ Aktionspunkte. Aber NIEMALS mit Duplikaten auffüllen — Qualität vor Quantität.
- Aktionspunkte: „Angebot an Firma X bis Freitag schicken" ist gut, „Follow-up durchführen" ist schlecht. Jede Aufgabe einzeln auflisten, NICHT zusammenfassen.
- KEINE DUPLIKATE: Wiederhole NIEMALS denselben Aktionspunkt. Jede Aufgabe darf GENAU EINMAL erscheinen. Wenn keine weiteren einzigartigen Aufgaben vorhanden sind, STOPPE — wiederhole NICHT bereits genannte Punkte.
- TERMINE: Extrahiere konkrete Termine wenn im Transkript genannt („bis Freitag", „nächste Woche", „Ende März", „in den nächsten Wochen"). Verwende „TBD" NUR wenn kein Zeitrahmen erwähnt wird.
- Entscheidu

In [ ]:
Make a copy and override the block you want to change
PROMPTS_MODIFIED = copy.deepcopy(PROMPTS)

# Uncomment ONE and paste your modified version:

# PROMPTS_MODIFIED["specificity_rules_en"] = """
# ## Quality Rules – MUST FOLLOW
#   - COMPLETENESS: Go through the ENTIRE transcript section by section. Do NOT skip any topics.
#   - MINIMUM COUNTS: Extract at least 8 action items and at least 5 decisions from a meeting over 30 minutes. For a 90-minute meeting, expect 15+ action items. But NEVER pad with duplicates — quality over quantity.
#   - Action items: are concrete, assigned tasks for a specific person (e.g., "Send proposal to Firma X by Friday"). List each task separately.
#   - Next steps: are broader project phases, upcoming events, or team-level dependencies (e.g., "Wait for vendor approval", "Schedule next meeting"). NEVER leave next_steps empty if the project's future was discussed.  
#   - NO DUPLICATES: NEVER repeat the same action item. Each task must appear EXACTLY ONCE. If you run out of unique action items, STOP — do not repeat earlier items.
#   - DUE DATES: Extract concrete dates when mentioned in the transcript ("by Friday", "next week", "end of March", "within several weeks"). Only use "TBD" when no timeframe is mentioned at all.
#   - Decisions: Only explicit agreements, not suggestions or possibilities.
#   - Use exact phrasing and quotes from the transcript when possible.
#   - Always use full person names as listed above. Do NOT invent names that don't appear in the transcript.
#   - Technical details: Include specific versions (e.g. "Ubuntu 24 LTS"), system names (e.g. "Bookstack", "GitLab"), configurations, storage sizes, RAM specs.
#   - Requirements: List each requirement separately with concrete specs, do NOT summarize as "Hardware requirements".
#   - THOROUGHNESS: Write detailed descriptions, not keywords. The summary should be at least 4 sentences. Every field should have substantial content.


# """


PROMPTS_MODIFIED["specificity_rules_en"] = """
## Quality Rules – MUST FOLLOW
  - COMPLETENESS: Go through the ENTIRE transcript section by section. Do NOT skip any topics.
  - MINIMUM COUNTS: Extract at least 8 action items and at least 5 decisions from a meeting over 30 minutes. For a 90-minute meeting, expect 15+ action items. But NEVER pad with duplicates — quality over quantity.
  - Action items: "Send proposal to Firma X by Friday" is good, "Follow up" is bad. List each task separately, do NOT merge them.
  - Next steps: are broader project phases or team-level dependencies. NEVER leave next_steps empty if the project's future was discussed.
  - NO DUPLICATES: NEVER repeat the same action item. Each task must appear EXACTLY ONCE.
  - DUE DATES: Extract concrete dates ("by Friday", "next week", "end of March"). Only use "TBD" when no timeframe is mentioned at all.
  - Decisions: Only explicit agreements, not suggestions or possibilities.
  - Use exact phrasing and quotes from the transcript when possible.
  - Always use full person names as listed above. Do NOT invent names.
  - Technical details: Include specific versions (e.g. "Ubuntu 24 LTS"), system names, configurations, storage sizes, RAM specs.
  - Requirements: List each requirement separately with concrete specs, do NOT summarize as "Hardware requirements".
  - THOROUGHNESS: Write detailed descriptions, not keywords. The summary should be at least 4 sentences.
  - ENTITY EXTRACTION: Treat every newly mentioned project, sub-task, tool, client, or person as a mandatory trigger. You MUST document each one separately in the output.
  - NO SILENT DROPS: Never discard a side-topic just because the discussion was brief. If a separate sub-project or task is discussed even for a few sentences, it MUST be preserved as its own distinct item.
  - NAMED PEOPLE CHECK: Every person mentioned by name in the transcript must appear in at least one field of the output. Do not omit any person.
  - NUANCE PRESERVATION: When a speaker qualifies a statement (e.g., "the algorithm is fine BUT the delay cause is unknown"), capture BOTH parts. Do NOT flatten qualified statements into one-sided conclusions.
  - PARTIAL ANSWERS: If a question was raised AND partially or fully answered in the same meeting, do NOT list it as an "open question." Instead, capture the answer given and note any remaining uncertainty.
  - KEY INSIGHTS: When a speaker explains a conceptual distinction, framework, or mental model (e.g., "MCP is different from API because..."), capture it in the summary or as a separate key_insight. Conceptual explanations are as important as action items.
  """

PROMPTS_MODIFIED["structure_analysis_en"] = """

You are an expert meeting analyst. Your task is to analyze the following transcript
  and produce a detailed structural outline.

  CRITICAL INSTRUCTIONS:
  1. LANGUAGE: Respond in English.
  2. Use confirmed speaker names, NOT speaker IDs.
  3. COMPLETENESS: Go through the ENTIRE transcript. Do NOT skip any topics, even brief side discussions.
  4. For long meetings, expect 8-15+ Topics.
  5. Output valid JSON only.
  6. ENTITY EXTRACTION: Every person, sub-project, tool, or administrative task mentioned — even briefly — must appear as its own topic or sub-item. Do NOT merge unrelated discussions.
  7. NO SILENT DROPS: A side-topic discussed for even 2-3 sentences must be its own topic entry, not absorbed into a broader topic.

  Produce a structural outline with:
  - meeting_title: Concise title
  - duration_estimate: Estimated duration
  - participants_active: List of active participants
  - topics: Array with topic_title, time_range, speakers_involved, summary, key_statements, decisions_made, action_items_mentioned, open_questions
  - cross_cutting_themes: Cross-cutting themes
  - overall_tone: Overall meeting tone

  Be THOROUGH and DETAILED. 
"""


PROMPTS_MODIFIED["json_formatting_open_source_en"] = """
STEP 1 — MANDATORY PRE-SCAN (complete before writing any JSON):
Go through the transcript and list every person, sub-project, tool, task, and topic.
This checklist is binding — every item MUST appear somewhere in the final JSON output.

STEP 2 — JSON OUTPUT:
Your response must be ONLY valid JSON.
Start with { and end with }.
No text, explanation, or markdown before or after the JSON.
Every item from Step 1 must be represented in the output.
"""



PROMPTS_MODIFIED["extraction_from_outline_en"] = """
You are an expert meeting protocol writer. You receive two inputs:
  1. A structural outline of the meeting (from a prior analysis)
  2. The original transcript as reference


  Your task: Produce the final meeting protocol in the specified JSON format.


  CRITICAL INSTRUCTIONS:
  1. LANGUAGE: Respond in the same language as the transcript.
  2. The outline gives you structure. The TRANSCRIPT is the ground truth — if the transcript contains a person, topic, or task NOT in the outline, you MUST still include it.
  3. SPECIFICITY: Concrete names, numbers, system names, dates. No generic descriptions.
  4. COMPLETENESS: Every topic from the outline must appear in at least one field of the protocol.
  5. Do NOT invent information not in the outline or transcript.
  6. Output valid JSON only.


  IMPORTANT: Be THOROUGH. Every topic from the outline must be covered. Short responses with only 2-3 action items are NOT acceptable.

  NUANCE RULES:
    - If the transcript contains qualified statements ("X is true, but Y"), preserve the full nuance. Do not simplify to just "X is true."
    - If someone explains a concept, framework, or distinction in detail, it MUST appear in the summary or a dedicated field. Do not reduce explanations to generic action items.
    - Open questions: Only list a question as "open" if it was genuinely left unanswered. If the speaker gave even a partial answer, capture the answer and note what remains uncertain.
"""




changed = [k for k in PROMPTS if PROMPTS[k] != PROMPTS_MODIFIED[k]]
if changed:
    for k in changed:
        print(f"Changed: {k}  ({len(PROMPTS[k])} -> {len(PROMPTS_MODIFIED[k])} chars)")
else:
    print("Nothing changed yet -- uncomment a block above and paste your edits.")# 

In [9]:
# Kopie der Prompts anlegen und nur ausgewählte Blöcke überschreiben
PROMPTS_MODIFIED = copy.deepcopy(PROMPTS)

PROMPTS_MODIFIED["specificity_rules_de"] = """
## Qualitätsregeln – UNBEDINGT EINHALTEN
  - VOLLSTÄNDIGKEIT: Gehe das GESAMTE Transkript Abschnitt für Abschnitt durch. Lass KEIN Thema aus.
  - MINDESTANZAHLEN: Extrahiere bei Meetings über 30 Minuten mindestens 8 Action Items und mindestens 5 Entscheidungen. Für ein 90‑minütiges Meeting sind 15+ Action Items zu erwarten. Fülle NIEMALS mit Duplikaten auf – Qualität vor Quantität.
  - Action Items: „Schicke das Angebot bis Freitag an Firma X“ ist gut, „Nachfassen“ ist schlecht. Führe jede Aufgabe separat auf, NICHT zusammenfassen.
  - Next Steps: sind größere Projektphasen oder teamweite Abhängigkeiten. Lasse das Feld next_steps NIEMALS leer, wenn über die Zukunft des Projekts gesprochen wurde.
  - KEINE DUPLIKATE: Wiederhole NIE dasselbe Action Item. Jede Aufgabe darf GENAU EINMAL vorkommen.
  - FÄLLIGKEITEN: Extrahiere konkrete Zeitangaben („bis Freitag“, „nächste Woche“, „Ende März“). Verwende „TBD“ nur, wenn keinerlei Zeithorizont erwähnt wird.
  - Entscheidungen: Nur explizite Vereinbarungen, keine Vorschläge oder Möglichkeiten.
  - Verwende, wo sinnvoll, die genaue Formulierung und Zitate aus dem Transkript.
  - Verwende immer vollständige Personennamen wie oben angegeben. Erfinde KEINE Namen.
  - Technische Details: Nenne konkrete Versionen (z.B. „Ubuntu 24 LTS“), Systemnamen, Konfigurationen, Speicherkapazitäten, RAM‑Größen.
  - Anforderungen: Führe jede Anforderung mit konkreten Spezifikationen separat auf, NICHT als allgemeines „Hardwareanforderungen“ zusammenfassen.
  - GRÜNDLICHKEIT: Schreibe ausführliche Beschreibungen, keine Schlagworte. Die Zusammenfassung soll mindestens 4 Sätze enthalten.
  - ENTITY EXTRACTION: Jede neu erwähnte Person, jedes Projekt, Teilprojekt, Tool oder jeder Kunde ist ein Pflicht‑Trigger. Du MUSST jede Entität separat im Output dokumentieren.
  - KEINE STILLSCHWEIGENDEN AUSLASSUNGEN: Verwirf ein Nebenthema nicht nur deshalb, weil es kurz besprochen wurde. Wenn ein separates Teilprojekt oder eine Aufgabe auch nur für wenige Sätze diskutiert wird, MUSS es als eigener Eintrag erhalten bleiben.
  - NAMENS-PRÜFUNG: Jede im Transkript namentlich erwähnte Person muss in mindestens einem Feld des Outputs vorkommen. Lass keine Person weg.
  - NUANCEN ERHALTEN: Wenn eine Aussage qualifiziert wird (z.B. „der Algorithmus ist okay, ABER die Ursache der Verzögerung ist unklar“), erfasse BEIDE Teile. Reduziere qualifizierte Aussagen NICHT auf einseitige Schlussfolgerungen.
  - TEILANTWORTEN: Wenn eine Frage gestellt UND im Meeting teilweise oder vollständig beantwortet wird, darf sie NICHT als „offene Frage“ erscheinen. Erfasse stattdessen die gegebene Antwort und notiere verbleibende Unsicherheit.
  - KERNERKENNTNISSE: Wenn jemand eine konzeptionelle Unterscheidung, ein Framework oder ein mentales Modell erklärt (z.B. „MCP unterscheidet sich von der API, weil …“), muss dies in der Zusammenfassung oder als eigenes key_insight auftauchen. Konzeptionelle Erklärungen sind genauso wichtig wie Action Items.
"""

PROMPTS_MODIFIED["structure_analysis_de"] = """
Du bist ein*e erfahrene*r Meeting‑Analyst*in. Deine Aufgabe ist es, das folgende Transkript
zu analysieren und eine detaillierte strukturelle Gliederung zu erstellen.

KRITISCHE ANWEISUNGEN:
  1. SPRACHE: Antworte auf Deutsch.
  2. Verwende bestätigte Sprechernamen, NICHT Speaker‑IDs.
  3. VOLLSTÄNDIGKEIT: Gehe das GESAMTE Transkript durch. Lass KEIN Thema aus, auch keine kurzen Nebendiskussionen.
  4. Für lange Meetings sind 8–15+ Themen (topics) zu erwarten.
  5. Gib ausschließlich gültiges JSON aus.
  6. ENTITY EXTRACTION: Jede Person, jedes Teilprojekt, jedes Tool oder jede organisatorische Aufgabe – selbst wenn nur kurz erwähnt – muss als eigenes Topic oder Sub‑Item erscheinen. Fasse UNVERWANDTE Diskussionen NICHT zusammen.
  7. KEINE STILLSCHWEIGENDEN AUSLASSUNGEN: Ein Nebenthema, das 2–3 Sätze lang diskutiert wird, braucht einen eigenen Topic‑Eintrag und darf nicht in einem Oberthema „verschwinden“.

Erzeuge eine strukturierte Gliederung mit:
  - meeting_title: Prägnanter Titel
  - duration_estimate: Geschätzte Dauer
  - participants_active: Liste der aktiven Teilnehmenden
  - topics: Array mit topic_title, time_range, speakers_involved, summary, key_statements, decisions_made, action_items_mentioned, open_questions
  - cross_cutting_themes: Querschnittsthemen
  - overall_tone: Gesamtstimmung des Meetings

Sei GRÜNDLICH und DETAILLIERT.
"""

PROMPTS_MODIFIED["json_formatting_open_source_de"] = """
SCHRITT 1 – VERPFLICHTENDER VORAB-SCAN (vor dem Schreiben von JSON abschließen):
Gehe das Transkript durch und liste jede Person, jedes Teilprojekt, jedes Tool, jede Aufgabe und jedes Thema auf.
Diese Checkliste ist bindend – jedes Element MUSS irgendwo im finalen JSON‑Output vorkommen.

SCHRITT 2 – JSON-AUSGABE:
Deine Antwort muss AUSSCHLIESSLICH aus gültigem JSON bestehen.
Beginne mit { und ende mit }.
Kein Text, keine Erklärung und kein Markdown vor oder nach dem JSON.
Jedes Element aus Schritt 1 muss im Output repräsentiert sein.
"""

PROMPTS_MODIFIED["extraction_from_outline_de"] = """
Du bist ein*e erfahrene*r Protokollführer*in. Du erhältst zwei Eingaben:
  1. Eine strukturelle Gliederung des Meetings (aus einer vorherigen Analyse)
  2. Das originale Transkript als Referenz

Deine Aufgabe: Erstelle das endgültige Sitzungsprotokoll im vorgegebenen JSON‑Format.

KRITISCHE ANWEISUNGEN:
  1. SPRACHE: Antworte in derselben Sprache wie das Transkript.
  2. Die Gliederung liefert dir die Struktur. Das TRANSKRIPT ist die Wahrheitsebene – wenn das Transkript eine Person, ein Thema oder eine Aufgabe enthält, die NICHT in der Gliederung stehen, musst du sie trotzdem aufnehmen.
  3. SPEZIFITÄT: Konkrete Namen, Zahlen, Systemnamen, Daten. Keine vagen Formulierungen.
  4. VOLLSTÄNDIGKEIT: Jedes Thema aus der Gliederung muss in mindestens einem Feld des Protokolls erscheinen.
  5. Erfinde KEINE Informationen, die weder in der Gliederung noch im Transkript vorkommen.
  6. Gib ausschließlich gültiges JSON aus.

WICHTIG: Sei GRÜNDLICH. Jedes Thema aus der Gliederung muss abgedeckt sein. Kurze Antworten mit nur 2–3 Action Items sind NICHT akzeptabel.

NUANZREGELN:
  - Wenn das Transkript qualifizierte Aussagen enthält („X stimmt, aber Y“), bewahre die gesamte Nuance. Reduziere das nicht auf „X stimmt“.
  - Wenn jemand ein Konzept, ein Framework oder eine Unterscheidung ausführlich erklärt, MUSS dies in der Zusammenfassung oder in einem eigenen Feld erscheinen. Reduziere solche Erklärungen nicht auf generische Action Items.
  - Offene Fragen: Führe eine Frage nur dann als „offen“, wenn sie tatsächlich unbeantwortet blieb. Wenn im Meeting eine (auch teilweise) Antwort gegeben wurde, erfasse die Antwort und beschreibe, was weiterhin unklar bleibt.
"""

changed = [k for k in PROMPTS if PROMPTS[k] != PROMPTS_MODIFIED[k]]
if changed:
    for k in changed:
        print(f"Changed: {k}  ({len(PROMPTS[k])} -> {len(PROMPTS_MODIFIED[k])} chars)")
else:
    print("Nothing changed yet -- uncomment a block above and paste your edits.")

Changed: specificity_rules_de  (1719 -> 2982 chars)
Changed: structure_analysis_de  (910 -> 1367 chars)
Changed: extraction_from_outline_de  (859 -> 1676 chars)
Changed: json_formatting_open_source_de  (247 -> 541 chars)


In [10]:
# Run with the modified prompt
modified_system = build_system_prompt(PROMPTS_MODIFIED, LANG, IS_CLIENT_MEETING)
print(modified_system[:] + "...ooooooooooooooooooooooooooooooooooooooooo" )
print(f"Running {MODEL} with modified prompt...")
start = time.time()
modified_output = call_model(modified_system, user_prompt, model=MODEL, max_tokens=8000)
modified_duration = round(time.time() - start, 2)

try:
    clean = modified_output.strip()
    if clean.startswith("```"):
        clean = clean.split("\n", 1)[1].rsplit("```", 1)[0]
    modified_protocol = json.loads(clean)
    print(f"Done in {modified_duration}s\n")
    print(json.dumps(modified_protocol, indent=2, ensure_ascii=False))
except json.JSONDecodeError as e:
    print(f"JSON parse error: {e}")
    print(modified_output)
    modified_protocol = None

You are an expert meeting protocol writer.

CRITICAL INSTRUCTIONS:
1. Respond in the SAME LANGUAGE as the transcript.
2. SPECIFICITY: Use concrete names, numbers, dates, and quotes from the transcript. Avoid generic descriptions.
3. Use the confirmed speaker names below, NOT the speaker IDs from the transcript.
4. Output valid JSON only.


You are an expert meeting protocol writer. You receive two inputs:
1. A structural outline of the meeting (from a prior analysis)
2. The original transcript as reference

Your task: Produce the final meeting protocol in the specified JSON format.

CRITICAL INSTRUCTIONS:
1. LANGUAGE: Respond in the same language as the transcript.
2. The outline is your primary source for structure and topics. The transcript is your source for exact quotes, names, and details.
3. SPECIFICITY: Concrete names, numbers, system names, dates. No generic descriptions.
4. COMPLETENESS: Every topic from the outline must appear in at least one field of the protocol.
5. Do NOT 

## 4. Compare

In [11]:
print("=" * 80)
print(f"BASELINE -- {baseline_duration}s")
print("=" * 80)
if baseline_protocol:
    print(json.dumps(baseline_protocol, indent=2, ensure_ascii=False))
else:
    print(baseline_output)

print(f"\n{'=' * 80}")
print(f"MODIFIED ({', '.join(changed) if changed else 'no changes'}) -- {modified_duration}s")
print("=" * 80)
if modified_protocol:
    print(json.dumps(modified_protocol, indent=2, ensure_ascii=False))
else:
    print(modified_output)

BASELINE -- 11.63s
{
  "meeting_protocol": {
    "title": "Podcast über Reisen und nachhaltiges Reisen",
    "date": "TBD",
    "participants": [
      "Anna",
      "Markus"
    ],
    "summary": "In diesem Podcast diskutierten Anna und Markus über die Faszination des Reisens, die Vorteile, die es mit sich bringt, und wie man sich auf Reisen vorbereitet. Sie sprachen über die kulturellen Erfahrungen, die man beim Reisen sammeln kann, und die Bedeutung von nachhaltigem Reisen. Anna und Markus teilten persönliche Erlebnisse und gaben Tipps, wie man umweltbewusst reisen und nachhaltige Souvenirs finden kann.",
    "action_items": [
      {
        "description": "Anna plant, Japan zu besuchen, um die Kultur und den Mount Fuji zu erleben.",
        "due_date": "TBD"
      },
      {
        "description": "Markus möchte Südamerika, insbesondere Peru und Machu Picchu, besuchen.",
        "due_date": "TBD"
      },
      {
        "description": "Anna und Markus empfehlen, Reiseversicherung

## 5. Save Results

In [ ]:
# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# name = Path(TRANSCRIPT_FILE).stem

# results = {
#     "transcript": name,
#     "language": LANG,
#     "model": MODEL,
#     "timestamp": timestamp,
#     "baseline": {
#         "output": baseline_output,
#         "protocol": baseline_protocol,
#         "duration": baseline_duration,
#     },
#     "modified": {
#         "output": modified_output,
#         "protocol": modified_protocol,
#         "duration": modified_duration,
#         "changed_blocks": changed,
#     },
# }

# outpath = Path(f"../results/{name}_{timestamp}.json")
# outpath.parent.mkdir(parents=True, exist_ok=True)
# outpath.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
# print(f"Saved to {outpath}")

Saved to ../results/meeting_08_20260409_131659.json
